In [1]:
import json
from pathlib import Path
import pandas as pd
import re

In [2]:
INPUT_DIR = Path("JSON_NO_STRUCTURE")
OUTPUT_DIR = Path("JSON_NO_STRUCTURE_EXTRACTED_TXT")
LOG_PATH = "json_to_txt_log.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
#for cleaning spacing
def clean_block_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)

    paragraphs = re.split(r"\n\s*\n", text)

    cleaned_paragraphs = []

    for paragraph in paragraphs:
        lines = [line.strip() for line in paragraph.splitlines() if line.strip()]

        if not lines:
            continue

        cleaned_paragraph = " ".join(lines)
        cleaned_paragraph = re.sub(r"\s{2,}", " ", cleaned_paragraph).strip()

        cleaned_paragraphs.append(cleaned_paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [4]:
def load_json_blocks(json_path):
    with open(json_path, "r", encoding="utf-8-sig") as file:
        data = json.load(file)

    if isinstance(data, dict):
        data = [data]

    return data


def blocks_to_text(blocks):
    ordered_blocks = []

    for original_position, block in enumerate(blocks):
        if not isinstance(block, dict):
            continue

        text = clean_block_text(str(block.get("text", "")))

        if not text:
            continue

        page = block.get("page", 0)
        block_id = block.get("block_id", original_position)

        try:
            page = int(page)
        except (TypeError, ValueError):
            page = 0

        try:
            block_id = int(block_id)
        except (TypeError, ValueError):
            block_id = original_position

        ordered_blocks.append(
            {
                "page": page,
                "block_id": block_id,
                "original_position": original_position,
                "text": text,
            }
        )

    ordered_blocks.sort(
        key=lambda block: (
            block["page"],
            block["block_id"],
            block["original_position"],
        )
    )

    return "\n\n".join(block["text"] for block in ordered_blocks).strip()


json_files = sorted(INPUT_DIR.rglob("*.json"))

print(f"Found {len(json_files)} JSON files in: {INPUT_DIR}")

log_rows = []
converted = 0
empty = 0
errors = 0

for index, json_path in enumerate(json_files, start=1):
    try:
        blocks = load_json_blocks(json_path)
        output_text = blocks_to_text(blocks)

        relative_path = json_path.relative_to(INPUT_DIR)
        output_path = OUTPUT_DIR / relative_path.with_suffix(".txt")
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if not output_text:
            empty += 1

            log_rows.append(
                {
                    "json_path": str(json_path),
                    "text_path": "",
                    "status": "empty",
                    "total_json_blocks": len(blocks),
                    "characters": 0,
                    "words": 0,
                    "error": "",
                }
            )

            print(f"EMPTY: {json_path}")
            continue

        output_path.write_text(output_text, encoding="utf-8")

        converted += 1

        log_rows.append(
            {
                "json_path": str(json_path),
                "text_path": str(output_path),
                "status": "ok",
                "total_json_blocks": len(blocks),
                "characters": len(output_text),
                "words": len(output_text.split()),
                "error": "",
            }
        )

        if index % 100 == 0:
            print(f"Processed {index}/{len(json_files)} files.")

    except Exception as error:
        errors += 1

        log_rows.append(
            {
                "json_path": str(json_path),
                "text_path": "",
                "status": "error",
                "total_json_blocks": "",
                "characters": 0,
                "words": 0,
                "error": str(error),
            }
        )

        print(f"ERROR: {json_path}: {error}")


log_df = pd.DataFrame(log_rows)
log_df.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")

print("\nConversion completed.")
print(f"Converted successfully: {converted}")
print(f"Empty outputs: {empty}")
print(f"Errors: {errors}")
print(f"Output folder: {OUTPUT_DIR}")

if not log_df.empty:
    display(
        log_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="files"))

Found 2363 JSON files in: JSON_NO_STRUCTURE
Processed 100/2363 files.
Processed 200/2363 files.
Processed 300/2363 files.
Processed 400/2363 files.
Processed 500/2363 files.
Processed 600/2363 files.
Processed 700/2363 files.
Processed 800/2363 files.
Processed 900/2363 files.
Processed 1000/2363 files.
Processed 1100/2363 files.
Processed 1200/2363 files.
Processed 1300/2363 files.
Processed 1400/2363 files.
Processed 1500/2363 files.
Processed 1600/2363 files.
Processed 1700/2363 files.
Processed 1800/2363 files.
Processed 1900/2363 files.
Processed 2000/2363 files.
Processed 2100/2363 files.
Processed 2200/2363 files.
Processed 2300/2363 files.

Conversion completed.
Converted successfully: 2363
Empty outputs: 0
Errors: 0
Output folder: JSON_NO_STRUCTURE_EXTRACTED_TXT


,status,files
0,ok,2363
